In [4]:
!unzip super-ai-engineer-season-6-ocr-2569.zip

Archive:  super-ai-engineer-season-6-ocr-2569.zip
replace final_data/images/constituency_10_1.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [6]:
!pip install -q openai tqdm pandas rapidfuzz easyocr

In [7]:
import pandas as pd
import easyocr
from rapidfuzz import process, fuzz
import re
import glob
import os
from tqdm.auto import tqdm

In [ ]:
import pandas as pd
import easyocr
from rapidfuzz import process, fuzz
import re
import glob
import os
from tqdm.auto import tqdm

print("⏳ กำลังโหลด EasyOCR Hybrid (V.3 + อัปเดต Template V.2)...")
reader = easyocr.Reader(['th', 'en'], gpu=True)

df_template = pd.read_csv('/content/final_data/submission_template_v3.csv')
if 'doc_id' not in df_template.columns:
    split_cols = df_template['id'].str.rsplit('_', n=1, expand=True)
    df_template['doc_id'] = split_cols[0]
    df_template['party_name'] = split_cols[1]

doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()
id_lookup = df_template.set_index(['doc_id', 'party_name'])['id'].to_dict()

def extract_last_number(text):
    t = text.replace('l', '1').replace('I', '1').replace('O', '0').replace('o', '0')
    t = t.replace('s', '5').replace('S', '5').replace('b', '6').replace('B', '8')
    t = re.sub(r'(?<=[0-9๐-๙,])ด', '1', t)
    t = re.sub(r'ด(?=[0-9๐-๙,])', '1', t)
    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    t = t.translate(thai_map)
    t_clean = re.sub(r'\s*,\s*', '', t)
    nums = re.findall(r'\d+', t_clean)
    if nums: return int(nums[-1])
    return 0

def fix_thai_typos(text):
    t = re.sub(r'[\.\-\*\_]', '', text).strip()
    if "เพื่อ" not in t: t = t.replace("เพือ", "เพื่อ")
    if "ภูมิใจ" not in t: t = t.replace("ภูมใจ", "ภูมิใจ")
    if "ประชา" not in t: t = t.replace("ประขา", "ประชา")
    if "ชาติ" not in t: t = t.replace("ชาต ", "ชาติ ")
    if "อนาคต" not in t: t = t.replace("อนาคด", "อนาคต")
    if "ใหม่" not in t: t = t.replace("ใหม", "ใหม่")
    if "พร้อม" not in t: t = t.replace("พรอม", "พร้อม")
    if "กรีน" not in t: t = t.replace("กรน", "กรีน").replace("กร์น", "กรีน")
    if "ไทย" not in t: t = t.replace("ทย", "ไทย")
    return t

def process_image_hybrid_v3(img_path, valid_parties, doc_type):
    result = reader.readtext(
        img_path, mag_ratio=2.0, contrast_ths=0.2, adjust_contrast=0.6
    )

    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda item: item['y'])
    grouped_lines = []
    current_group = []
    y_threshold = 25 if doc_type == 'party_list' else 35

    for item in lines:
        if not current_group:
            current_group.append(item)
        else:
            if abs(item['y'] - current_group[-1]['y']) < y_threshold:
                current_group.append(item)
            else:
                current_group.sort(key=lambda i: i['x'])
                grouped_lines.append(" ".join([i['text'] for i in current_group]))
                current_group = [item]

    if current_group:
        current_group.sort(key=lambda i: i['x'])
        grouped_lines.append(" ".join([i['text'] for i in current_group]))

    votes_dict = {}
    for line in grouped_lines:
        vote_num = extract_last_number(line)
        if vote_num == 0: continue

        text_no_num = re.sub(r'[\d๐-๙,]', '', line).strip()
        text_fixed = fix_thai_typos(text_no_num)

        best_match = process.extractOne(text_fixed, valid_parties, scorer=fuzz.token_set_ratio)

        if best_match and best_match[1] > 50:
            matched_party = best_match[0]
            votes_dict[matched_party] = max(votes_dict.get(matched_party, 0), vote_num)

    return votes_dict

# ลุยรันทั้งโฟลเดอร์ 847 ไฟล์


file_list = sorted(glob.glob("/content/final_data/images/*.png"))
final_submission = {row_id: 0 for row_id in df_template['id']}

print(f"🚀 เริ่มสแกนระบบ Hybrid V.3 {len(file_list)} ไฟล์ ด้วย Template ใหม่...")

for img_path in tqdm(file_list):
    filename = os.path.basename(img_path)
    doc_id = re.sub(r'_page\d+$', '', filename.replace('.png', ''))

    doc_type = "party_list" if "party_list" in filename else "constituency"

    # 🎯 ใช้ Template ใหม่ในการดึงชื่อพรรค
    valid_parties = doc_to_parties.get(doc_id, [])
    if not valid_parties: continue

    extracted_votes = process_image_hybrid_v3(img_path, valid_parties, doc_type)

    for party, vote in extracted_votes.items():
        row_id = id_lookup.get((doc_id, party))
        if row_id:
            final_submission[row_id] = max(final_submission[row_id], vote)

# เซฟผลลัพธ์เป็นฟอร์แมต 2 คอลัมน์พร้อมส่ง
df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
df_out.to_csv("/content/submission_final_v2.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print("\n" + "="*40)
print(f"✅ ปิดจ๊อบสมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_final_v2.csv")
print("="*40)

⏳ กำลังโหลด EasyOCR Hybrid (V.3 + อัปเดต Template V.2)...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete🚀 เริ่มสแกนระบบ Hybrid V.3 846 ไฟล์ ด้วย Template ใหม่...


  0%|          | 0/846 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume

In [ ]:
import easyocr
import cv2
import matplotlib.pyplot as plt

# 1. ระบุรูป Party List ที่มีปัญหา (ลองเปลี่ยนชื่อไฟล์ดูครับ)
test_image = "/content/data/images/constituency_10_10.png"

reader = easyocr.Reader(['th', 'en'], gpu=True)
result = reader.readtext(test_image)

print("--- สิ่งที่ EasyOCR มองเห็นแบบดิบๆ ---")
lines = []
for bbox, text, _ in result:
    y_center = (bbox[0][1] + bbox[2][1]) / 2
    lines.append({'text': text, 'y': y_center})

lines.sort(key=lambda x: x['y'])

# ลองพิมพ์ออกมาดู 20 บรรทัดแรก
for i, line in enumerate(lines[:20]):
    print(f"แกน Y: {line['y']:.1f} | ข้อความ: {line['text']}")

In [ ]:
import easyocr

# 1. ลองหาไฟล์ที่เป็น Party List แบบ "หน้า 2" หรือ "หน้า 3"
# (เพราะหน้า 1 มักจะเป็นหัวกระดาษ หน้า 2-3 จะเป็นตารางพรรคอัดแน่นๆ ครับ)
test_image = "/content/data/images/constituency_10_10_page2.png" # ลองเปลี่ยนไฟล์ดูครับ

reader = easyocr.Reader(['th', 'en'], gpu=True)
result = reader.readtext(test_image)

lines = []
for bbox, text, _ in result:
    y_center = (bbox[0][1] + bbox[2][1]) / 2
    lines.append({'text': text, 'y': y_center})

lines.sort(key=lambda x: x['y'])

print("--- สิ่งที่ EasyOCR มองเห็นในโซนตารางคะแนน ---")
# ขอส่องบรรทัดที่ 15 ถึง 45 (ช่วงที่เป็นรายชื่อพรรค)
for i, line in enumerate(lines[15:45]):
    print(f"แกน Y: {line['y']:.1f} | ข้อความ: {line['text']}")

In [ ]:
import easyocr
import glob
import os

print("🔍 กำลังค้นหาไฟล์ Party List แบบอัตโนมัติ...")
# ล็อกเป้าหาหน้า Party List หน้า 2 (ที่มีตารางพรรคแน่นๆ)
party_list_files = glob.glob("/content/data/images/party_list_*_page2.png")

if party_list_files:
    test_image = party_list_files[0]
    print(f"🎯 เจอไฟล์เป้าหมายแล้ว: {os.path.basename(test_image)}")

    reader = easyocr.Reader(['th', 'en'], gpu=True)
    result = reader.readtext(test_image, mag_ratio=2.0)

    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda x: x['y'])

    print("\n--- สิ่งที่ AI มองเห็น (สังเกตแกน X ว่ามันมีคอลัมน์ซ้าย-ขวาไหม) ---")
    # ปริ้นท์ 30 บรรทัดแรกมาส่องกันครับ
    for line in lines[:30]:
        print(f"แกน Y: {line['y']:<6.1f} | แกน X: {line['x']:<6.1f} | {line['text']}")
else:
    print("❌ หาไฟล์ไม่เจอครับ ตรวจสอบ Path อีกครั้ง")

In [ ]:
import cv2
import numpy as np
import glob
import os
from tqdm.auto import tqdm

def has_table(img_path, min_area_ratio=0.10):
    """
    ฟังก์ชันเช็คว่าภาพนี้เป็น 'หน้าตาราง' หรือไม่
    min_area_ratio = 0.10 หมายถึง ตารางต้องใหญ่เกิน 10% ของพื้นที่กระดาษ
    """
    # 1. โหลดภาพด้วย OpenCV
    img = cv2.imread(img_path)
    if img is None:
        return False

    height, width = img.shape[:2]
    image_area = height * width

    # แปลงเป็นสีเทา
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. แปลงเป็นภาพขาวดำ (เน้นเส้นให้เป็นสีขาว พื้นเป็นสีดำ)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)

    # 3. สร้าง Kernel (ไม้บรรทัด) เพื่อจับเส้นตั้งและเส้นนอน
    # กำหนดให้เส้นต้องยาวอย่างน้อย 1/40 ของขนาดภาพ ถึงจะนับว่าเป็นเส้นตาราง
    h_len = width // 40
    v_len = height // 40

    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (h_len, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, v_len))

    # 4. สกัดเฉพาะเส้นออกมา (ตัวหนังสือจะหายไปหมด)
    horizontal_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    vertical_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel, iterations=2)

    # 5. ประกอบเส้นตั้งและเส้นนอนเข้าด้วยกันเป็น Grid Mask
    table_mask = cv2.addWeighted(horizontal_lines, 0.5, vertical_lines, 0.5, 0.0)
    _, table_mask = cv2.threshold(table_mask, 50, 255, cv2.THRESH_BINARY)

    # 6. หาโครงร่าง (Contours) ของตาราง
    contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return False # ไม่มีเส้นอะไรเลย = ไม่ใช่ตาราง

    # 7. หากล่องตารางที่ใหญ่ที่สุด
    max_table_area = 0
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        max_table_area = max(max_table_area, w * h)

    # 8. ตัดสินใจ: ถ้าตารางที่ใหญ่ที่สุด กินพื้นที่เกิน 10% ของกระดาษ ถือว่าสอบผ่าน!
    if (max_table_area / image_area) > min_area_ratio:
        return True

    return False

# ==========================================
# 🚀 ลองใช้งานจริงเพื่อคัดกรองไฟล์
# ==========================================
all_files = sorted(glob.glob("/content/data/images/*.png"))
table_files = []
garbage_files = []

print(f"🔍 เริ่มสแกนคัดกรองหน้ากระดาษจากทั้งหมด {len(all_files)} ไฟล์...")

for img_path in tqdm(all_files):
    if has_table(img_path):
        table_files.append(img_path)
    else:
        garbage_files.append(img_path)

print("\n" + "="*40)
print(f"📊 สรุปผลการคัดกรอง:")
print(f"✅ เจอหน้าตาราง (นำไปใช้ต่อ): {len(table_files)} ไฟล์")
print(f"🗑️ เจอหน้าปก/ลายเซ็น (คัดทิ้ง): {len(garbage_files)} ไฟล์")
print("="*40)

# ตอนนี้สามารถเอา list `table_files` ไปใช้ในลูป OCR ต่อได้เลยครับ!

In [ ]:
!pip install pytesseract

In [ ]:
import cv2
import easyocr
import pandas as pd
import re
import os
import glob
from rapidfuzz import process, fuzz
from tqdm.auto import tqdm

print("⏳ กำลังเตรียมระบบ God Mode (OpenCV Cropper + EasyOCR)...")
reader = easyocr.Reader(['th', 'en'], gpu=True)

# 1. โหลด Template
df_template = pd.read_csv('/content/data/submission_template.csv')
doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()
id_lookup = df_template.set_index(['doc_id', 'party_name'])['id'].to_dict()

def extract_last_number(text):
    t = text.replace('l', '1').replace('I', '1').replace('O', '0').replace('o', '0')
    t = t.replace('s', '5').replace('S', '5').replace('b', '6').replace('B', '8')
    t = re.sub(r'(?<=[0-9๐-๙,])ด', '1', t)
    t = re.sub(r'ด(?=[0-9๐-๙,])', '1', t)
    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    t = t.translate(thai_map)
    t_clean = re.sub(r'\s*,\s*', '', t)
    nums = re.findall(r'\d+', t_clean)
    if nums:
        return int(nums[-1])
    return 0

def process_image_god_mode(img_path, valid_parties, doc_type):
    # --- 1. กรรไกรอัจฉริยะตัดตารางด้วย OpenCV ---
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)

    h_len, v_len = img.shape[1]//40, img.shape[0]//40
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (h_len, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, v_len))

    h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel, iterations=2)
    table_mask = cv2.addWeighted(h_lines, 0.5, v_lines, 0.5, 0.0)

    contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    max_area = 0
    table_bbox = None
    for c in contours:
        area = cv2.contourArea(c)
        if area > 50000:
            x, y, w, h = cv2.boundingRect(c)
            if area > max_area:
                max_area = area
                table_bbox = (x, y, w, h)

    if table_bbox:
        x, y, w, h = table_bbox
        pad = 20 # เผื่อขอบ 20px ให้ EasyOCR หายใจ
        cropped_img = img[max(0, y-pad) : y+h+pad, max(0, x-pad) : x+w+pad]
    else:
        cropped_img = img # ถ้าไม่มีตาราง (เผื่อหลุดมา) ก็ส่งรูปเต็ม

    # --- 2. ส่งรูปที่ตัดแล้วให้ EasyOCR อ่าน ---
    # โยน Numpy Array (cropped_img) เข้าไปตรงๆ ได้เลย!
    result = reader.readtext(
        cropped_img,
        mag_ratio=2.0,
        contrast_ths=0.2,
        adjust_contrast=0.6
    )

    # --- 3. จัดกลุ่มบรรทัด (ตาข่ายอัจฉริยะ) ---
    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda item: item['y'])
    grouped_lines = []
    current_group = []

    y_threshold = 25 if doc_type == 'party_list' else 35

    for item in lines:
        if not current_group:
            current_group.append(item)
        else:
            if abs(item['y'] - current_group[-1]['y']) < y_threshold:
                current_group.append(item)
            else:
                current_group.sort(key=lambda i: i['x'])
                grouped_lines.append(" ".join([i['text'] for i in current_group]))
                current_group = [item]

    if current_group:
        current_group.sort(key=lambda i: i['x'])
        grouped_lines.append(" ".join([i['text'] for i in current_group]))

    # --- 4. สกัดคะแนน ---
    votes_dict = {}
    for line in grouped_lines:
        vote_num = extract_last_number(line)
        if vote_num == 0: continue

        text_no_num = re.sub(r'[\d๐-๙,]', '', line).strip()
        best_match = process.extractOne(text_no_num, valid_parties, scorer=fuzz.token_set_ratio)

        if best_match and best_match[1] > 50:
            matched_party = best_match[0]
            votes_dict[matched_party] = max(votes_dict.get(matched_party, 0), vote_num)

    return votes_dict

# ==========================================
# 🚀 5. ลุยรันกับไฟล์ 599 ไฟล์ที่คัดกรองแล้ว
# ==========================================
final_submission = {row_id: 0 for row_id in df_template['id']}

# วนลูปเฉพาะใน table_files (599 ไฟล์ที่คัดไว้)
print(f"🚀 เริ่มกวาดตารางและสกัดคะแนน {len(table_files)} ไฟล์...")
for img_path in tqdm(table_files):
    filename = os.path.basename(img_path)
    doc_id = re.sub(r'_page\d+$', '', filename.replace('.png', ''))

    doc_type = "party_list" if "party_list" in doc_id else "constituency"
    valid_parties = doc_to_parties.get(doc_id, [])
    if not valid_parties: continue

    extracted_votes = process_image_god_mode(img_path, valid_parties, doc_type)

    for party, vote in extracted_votes.items():
        row_id = id_lookup.get((doc_id, party))
        if row_id:
            final_submission[row_id] = max(final_submission[row_id], vote)

# เซฟผลลัพธ์
df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
df_out.to_csv("/content/submission_god_mode.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print("\n" + "="*40)
print(f"✅ ปิดจ๊อบ God Mode สมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_god_mode.csv")
print("="*40)

In [ ]:
import pandas as pd

print("🔍 เริ่มกระบวนการชันสูตรข้อมูล (Error Analysis) V.2...")

# 1. โหลดข้อมูล Hybrid ของเรา (ตัวที่ดีที่สุด)
df_hybrid = pd.read_csv('/content/submission_hybrid.csv')

# 2. โหลด Template ข้อมูล
df_template = pd.read_csv('/content/data/submission_template.csv')

# 🎯 จุดที่แก้บั๊ก: สั่งเตะคอลัมน์ 'votes' ของ Template ทิ้งไปก่อน (ถ้ามี) จะได้ไม่ชนกัน!
df_template_clean = df_template.drop(columns=['votes'], errors='ignore')

# 3. รวมร่าง (Merge) ตอนนี้จะมีคอลัมน์ votes แค่อันเดียวแล้ว
df_merged = pd.merge(df_hybrid, df_template_clean, on='id', how='left')

# 4. กรองเอาเฉพาะพวกที่ได้ 0 คะแนน (ตอนนี้เรียก 'votes' ได้ตามปกติแล้ว)
df_zero = df_merged[df_merged['votes'] == 0]

# 5. แยกวิเคราะห์ระหว่าง ส.ส.เขต และ ปาร์ตี้ลิสต์
df_zero_constituency = df_zero[df_zero['id'].str.startswith('constituency')]
df_zero_partylist = df_zero[df_zero['id'].str.startswith('party_list')]

print("\n" + "="*40)
print(f"📊 สรุปยอดผู้สูญหายทั้งหมด: {len(df_zero)} แถว")
print(f"   - ส.ส.เขต: {len(df_zero_constituency)} แถว")
print(f"   - ปาร์ตี้ลิสต์: {len(df_zero_partylist)} แถว")
print("="*40)

# 6. ไฮไลท์สำคัญ: พรรคไหนในปาร์ตี้ลิสต์ที่หายไปมากที่สุด 15 อันดับแรก!
print("\n🚨 Top 15 พรรคปาร์ตี้ลิสต์ที่ AI อ่านไม่ออกมากที่สุด:")
top_missing_parties = df_zero_partylist['party_name'].value_counts().head(15)
for party, count in top_missing_parties.items():
    print(f"   ❌ {party}: หายไป {count} ครั้ง")

In [ ]:
import pandas as pd
import easyocr
from rapidfuzz import process, fuzz
import re
import glob
import os
from tqdm.auto import tqdm

print("⏳ กำลังโหลด EasyOCR Hybrid (V.2 ระบบพจนานุกรมแก้คำผิด)...")
reader = easyocr.Reader(['th', 'en'], gpu=True)

# โหลด Template
df_template = pd.read_csv('/content/submission_template.csv')
doc_to_parties = df_template.groupby("doc_id")["party_name"].apply(list).to_dict()
id_lookup = df_template.set_index(['doc_id', 'party_name'])['id'].to_dict()

def extract_last_number(text):
    """ฟังก์ชันดึงตัวเลข (ปราบภาษาอังกฤษปลอมตัว + ด.เด็ก)"""
    t = text.replace('l', '1').replace('I', '1').replace('O', '0').replace('o', '0')
    t = t.replace('s', '5').replace('S', '5').replace('b', '6').replace('B', '8')
    t = re.sub(r'(?<=[0-9๐-๙,])ด', '1', t)
    t = re.sub(r'ด(?=[0-9๐-๙,])', '1', t)

    thai_map = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    t = t.translate(thai_map)
    t_clean = re.sub(r'\s*,\s*', '', t)

    nums = re.findall(r'\d+', t_clean)
    if nums:
        return int(nums[-1])
    return 0

def fix_thai_typos(text):
    """🎯 ด่านกักกันโรค: ซ่อมคำผิดยอดฮิตก่อนส่งไปเทียบชื่อ"""
    # ซ่อมพวกสระ/วรรณยุกต์หาย
    t = text.replace("เพือ", "เพื่อ").replace("ภูมใจ", "ภูมิใจ")
    t = t.replace("ชาต ", "ชาติ ").replace("ประขา", "ประชา")

    # ซ่อมพรรคชื่อสั้น และพรรคยอดฮิตที่ตกหล่น
    t = t.replace("ใหม", "ใหม่").replace("พรอม", "พร้อม")
    t = t.replace("กรน", "กรีน").replace("กร์น", "กรีน")
    t = t.replace("ทย", "ไทย")
    t = t.replace("อนาคด", "อนาคต")

    # เอาพวกสัญลักษณ์ขยะที่ชอบปนมากับชื่อพรรคออก
    t = re.sub(r'[\.\-\*\_]', '', t)
    return t

def process_image_hybrid_v2(img_path, valid_parties, doc_type):
    result = reader.readtext(
        img_path,
        mag_ratio=2.0,
        contrast_ths=0.2,
        adjust_contrast=0.6
    )

    lines = []
    for bbox, text, _ in result:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2
        lines.append({'text': text, 'y': y_center, 'x': x_center})

    lines.sort(key=lambda item: item['y'])
    grouped_lines = []
    current_group = []

    y_threshold = 25 if doc_type == 'party_list' else 35

    for item in lines:
        if not current_group:
            current_group.append(item)
        else:
            if abs(item['y'] - current_group[-1]['y']) < y_threshold:
                current_group.append(item)
            else:
                current_group.sort(key=lambda i: i['x'])
                grouped_lines.append(" ".join([i['text'] for i in current_group]))
                current_group = [item]

    if current_group:
        current_group.sort(key=lambda i: i['x'])
        grouped_lines.append(" ".join([i['text'] for i in current_group]))

    votes_dict = {}
    for line in grouped_lines:
        vote_num = extract_last_number(line)
        if vote_num == 0: continue

        # เรียกใช้ตัวแก้คำผิด
        text_no_num = re.sub(r'[\d๐-๙,]', '', line).strip()
        text_fixed = fix_thai_typos(text_no_num)

        # ปรับให้ RapidFuzz ใจดีขึ้น
        best_match = process.extractOne(text_fixed, valid_parties, scorer=fuzz.token_set_ratio)

        if best_match and best_match[1] > 45:
            matched_party = best_match[0]
            votes_dict[matched_party] = max(votes_dict.get(matched_party, 0), vote_num)

    return votes_dict

# ==========================================
# 🚀 ลุยรันทั้งโฟลเดอร์แบบ 100%
# ==========================================
file_list = sorted(glob.glob("/content/data/images/*.png"))
final_submission = {row_id: 0 for row_id in df_template['id']}

print(f"🚀 เริ่มสแกนระบบ Hybrid V.2 {len(file_list)} ไฟล์...")

for img_path in tqdm(file_list):
    filename = os.path.basename(img_path)
    doc_id = re.sub(r'_page\d+$', '', filename.replace('.png', ''))

    doc_type = "party_list" if "party_list" in filename else "constituency"
    valid_parties = doc_to_parties.get(doc_id, [])
    if not valid_parties: continue

    extracted_votes = process_image_hybrid_v2(img_path, valid_parties, doc_type)

    for party, vote in extracted_votes.items():
        row_id = id_lookup.get((doc_id, party))
        if row_id:
            final_submission[row_id] = max(final_submission[row_id], vote)

# เซฟผลลัพธ์
df_out = pd.DataFrame(list(final_submission.items()), columns=['id', 'votes'])
df_out.to_csv("/content/submission_hybrid.csv", index=False)

filled_rows = df_out['votes'].gt(0).sum()
print("\n" + "="*40)
print(f"✅ ปิดจ๊อบ Hybrid V.2 สมบูรณ์! เติมคะแนนได้ {filled_rows}/{len(df_out)} แถว")
# 🎯 แก้ชื่อไฟล์ตรงนี้ เพื่อไม่ให้โหลดผิดตัวครับ!
print("📂 เซฟไฟล์พร้อมส่ง: /content/submission_600176.csv")
print("="*40)